# Proyek Analisis Data: Air Quality Dataset

**Nama:** Muhammad Putra Harifin Pane  
**Email:** putraharifin@gmail.com  
**ID Dicoding:** putraharifin25  

## Gambaran Umum

Proyek ini menganalisis data kualitas udara dari 12 stasiun pemantauan di Beijing selama periode Maret 2013 sampai Februari 2017. Fokus utama analisis adalah **PM2.5** sebagai indikator utama polusi udara, perbedaan kondisi antarstasiun, pola waktu, serta hubungan PM2.5 dengan faktor meteorologi.

> **Catatan:** notebook ini membaca seluruh file `PRSA_Data_*.csv` dari folder `data/`. Pastikan 12 file dataset ditempatkan di folder tersebut sebelum menjalankan notebook.


## Menentukan Pertanyaan Bisnis

Pertanyaan dirumuskan dengan pendekatan **SMART (Specific, Measurable, Action-oriented, Relevant, Time-bound)**.

### Pertanyaan 1
**Bagaimana perbedaan rata-rata konsentrasi PM2.5 bulanan antar-12 stasiun pemantauan Beijing selama periode Maret 2013–Februari 2017, dan stasiun mana yang memiliki rata-rata PM2.5 tertinggi?**

- **Specific:** fokus pada PM2.5 dan 12 stasiun.
- **Measurable:** menggunakan rata-rata PM2.5 bulanan dan rata-rata keseluruhan per stasiun.
- **Action-oriented:** hasil dapat digunakan untuk memprioritaskan stasiun/wilayah yang membutuhkan perhatian lebih.
- **Relevant:** PM2.5 merupakan indikator penting untuk evaluasi kualitas udara.
- **Time-bound:** Maret 2013–Februari 2017.

### Pertanyaan 2
**Bagaimana hubungan kondisi meteorologi (suhu/TEMP, tekanan/PRES, titik embun/DEWP, curah hujan/RAIN, dan kecepatan angin/WSPM) dengan rata-rata PM2.5 selama Maret 2013–Februari 2017?**

- **Specific:** fokus pada PM2.5 dan lima variabel meteorologi.
- **Measurable:** menggunakan koefisien korelasi dan perbandingan PM2.5 pada kelompok kondisi cuaca.
- **Action-oriented:** membantu menentukan faktor cuaca yang perlu dipertimbangkan saat pemantauan/peringatan kualitas udara.
- **Relevant:** kondisi meteorologi dapat memengaruhi penyebaran dan akumulasi polutan.
- **Time-bound:** Maret 2013–Februari 2017.


## Import Semua Packages/Library yang Digunakan

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

sns.set_theme(style="whitegrid", context="notebook")

DATA_DIR = Path("data")
CSV_FILES = sorted(DATA_DIR.glob("PRSA_Data_*.csv"))

print(f"Jumlah file CSV ditemukan: {len(CSV_FILES)}")
if len(CSV_FILES) != 12:
    raise FileNotFoundError(
        "Pastikan tepat 12 file PRSA_Data_*.csv tersedia di folder data/. "
        f"Saat ini ditemukan {len(CSV_FILES)} file."
    )


## Data Wrangling

### Gathering Data

Seluruh 12 dataset stasiun dibaca dan digabungkan menjadi satu DataFrame. Kolom `station` dipertahankan sebagai identitas stasiun.


In [ ]:
dfs = []

for file in CSV_FILES:
    temp = pd.read_csv(file)
    temp["source_file"] = file.name
    dfs.append(temp)

df = pd.concat(dfs, ignore_index=True)

print("Ukuran data gabungan:", df.shape)
print("Jumlah stasiun:", df["station"].nunique())
print("Daftar stasiun:", sorted(df["station"].dropna().unique()))
display(df.head())


### Assessing Data

Beberapa masalah kualitas data yang diperiksa:

1. **Missing value**, terutama pada variabel polutan seperti PM2.5, PM10, SO2, NO2, CO, dan O3.
2. **Duplicate data**, untuk memastikan tidak ada observasi yang tercatat lebih dari sekali.
3. **Invalid/inconsistent value**, terutama nilai numerik yang tidak masuk akal dan konsistensi timestamp.
4. **Outlier**, terutama pada PM2.5 dan variabel meteorologi. Outlier tidak otomatis dihapus karena dapat merepresentasikan episode polusi ekstrem yang memang penting secara analitis.


In [ ]:
print("=== INFO DATA ===")
df.info()

print("\n=== MISSING VALUE ===")
missing = df.isna().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)
missing_table = pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})
display(missing_table)

print("\n=== DUPLICATE ===")
print("Jumlah duplicate penuh:", df.duplicated().sum())

print("\n=== RANGE NUMERIK UTAMA ===")
numeric_cols = ["PM2.5", "PM10", "SO2", "NO2", "CO", "O3", "TEMP", "PRES", "DEWP", "RAIN", "WSPM"]
display(df[numeric_cols].describe().T)

print("\n=== CEK NILAI NEGATIF ===")
negative_counts = (df[numeric_cols] < 0).sum().sort_values(ascending=False)
display(negative_counts[negative_counts > 0])

print("\n=== CEK KONSISTENSI STASIUN ===")
display(df["station"].value_counts().sort_index())


**Steps to Take:**

- Menghapus duplicate penuh jika ditemukan.
- Membentuk timestamp dari `year`, `month`, `day`, dan `hour`.
- Menjadikan timestamp sebagai dasar pengurutan data.
- Missing value numerik akan diinterpolasi berdasarkan urutan waktu dalam masing-masing stasiun, kemudian sisa missing value diisi dengan median stasiun.
- Missing value `wd` akan diisi menggunakan nilai sebelumnya/berikutnya dalam stasiun; jika masih kosong, digunakan modus stasiun.
- Nilai negatif pada variabel yang secara definisi tidak boleh negatif akan diperlakukan sebagai missing dan kemudian diimputasi.
- Outlier tidak dihapus secara otomatis karena episode polusi ekstrem merupakan informasi yang relevan.


### Cleaning Data

In [ ]:
df_clean = df.copy()

# 1. Hapus duplicate penuh
before = len(df_clean)
df_clean = df_clean.drop_duplicates().copy()
print(f"Duplicate yang dihapus: {before - len(df_clean)}")

# 2. Bentuk timestamp
df_clean["datetime"] = pd.to_datetime(
    df_clean[["year", "month", "day", "hour"]],
    errors="coerce"
)

# 3. Nilai yang secara logika tidak boleh negatif -> NaN
non_negative_cols = ["PM2.5", "PM10", "SO2", "NO2", "CO", "O3", "PRES", "RAIN", "WSPM"]
for col in non_negative_cols:
    df_clean.loc[df_clean[col] < 0, col] = np.nan

# 4. Urutkan berdasarkan stasiun dan waktu
df_clean = df_clean.sort_values(["station", "datetime"]).reset_index(drop=True)

# 5. Imputasi numerik berbasis waktu per stasiun
numeric_cols = ["PM2.5", "PM10", "SO2", "NO2", "CO", "O3", "TEMP", "PRES", "DEWP", "RAIN", "WSPM"]

for col in numeric_cols:
    df_clean[col] = (
        df_clean.groupby("station")[col]
        .transform(lambda s: s.interpolate(method="linear", limit_direction="both"))
    )
    df_clean[col] = df_clean[col].fillna(df_clean.groupby("station")[col].transform("median"))

# 6. Imputasi arah angin
df_clean["wd"] = (
    df_clean.groupby("station")["wd"]
    .transform(lambda s: s.ffill().bfill())
)
wd_mode = df_clean["wd"].mode().iloc[0]
df_clean["wd"] = df_clean["wd"].fillna(wd_mode)

# 7. Kolom turunan waktu
df_clean["year_month"] = df_clean["datetime"].dt.to_period("M").astype(str)
df_clean["month_num"] = df_clean["datetime"].dt.month
df_clean["season"] = df_clean["month_num"].map({
    12: "Winter", 1: "Winter", 2: "Winter",
    3: "Spring", 4: "Spring", 5: "Spring",
    6: "Summer", 7: "Summer", 8: "Summer",
    9: "Autumn", 10: "Autumn", 11: "Autumn"
})

print("Missing value setelah cleaning:")
display(df_clean.isna().sum().sort_values(ascending=False).head(10))
print("Ukuran data setelah cleaning:", df_clean.shape)


**Insight:**

- Data dari seluruh stasiun berhasil disatukan dalam satu struktur analisis.
- Missing value ditangani tanpa langsung menghapus baris agar deret waktu tetap terjaga.
- Nilai ekstrem dipertahankan karena dapat menggambarkan episode polusi yang penting untuk analisis kualitas udara.


## Exploratory Data Analysis (EDA)

### Eksplorasi 1 — Statistik PM2.5 per Stasiun

In [ ]:
station_pm25 = (
    df_clean.groupby("station")["PM2.5"]
    .agg(["count", "mean", "median", "std", "min", "max"])
    .sort_values("mean", ascending=False)
)

display(station_pm25)

monthly_pm25 = (
    df_clean.groupby(["year_month", "station"], as_index=False)["PM2.5"]
    .mean()
    .rename(columns={"PM2.5": "PM2.5_mean"})
)

station_ranking = (
    df_clean.groupby("station", as_index=False)["PM2.5"]
    .mean()
    .sort_values("PM2.5", ascending=False)
)

display(station_ranking)


### Eksplorasi 2 — Hubungan PM2.5 dengan Faktor Meteorologi


In [ ]:
weather_cols = ["TEMP", "PRES", "DEWP", "RAIN", "WSPM"]
corr_cols = ["PM2.5"] + weather_cols

corr_matrix = df_clean[corr_cols].corr()
display(corr_matrix)

season_pm25 = (
    df_clean.groupby("season")["PM2.5"]
    .agg(["mean", "median", "std", "count"])
    .reindex(["Spring", "Summer", "Autumn", "Winter"])
)

display(season_pm25)


## Visualization & Explanatory Analysis

### Pertanyaan 1
**Bagaimana perbedaan rata-rata konsentrasi PM2.5 bulanan antar-12 stasiun selama Maret 2013–Februari 2017, dan stasiun mana yang memiliki rata-rata PM2.5 tertinggi?**


In [ ]:
# Visualisasi 1: ranking rata-rata PM2.5 per stasiun
plt.figure(figsize=(12, 6))
plot_data = station_ranking.sort_values("PM2.5", ascending=True)

ax = sns.barplot(
    data=plot_data,
    x="PM2.5",
    y="station",
    errorbar=None
)
ax.set_title("Rata-rata Konsentrasi PM2.5 per Stasiun (2013–2017)")
ax.set_xlabel("Rata-rata PM2.5")
ax.set_ylabel("Stasiun")
plt.tight_layout()
plt.show()


In [ ]:
# Visualisasi 2: tren rata-rata PM2.5 bulanan semua stasiun
monthly_all = (
    df_clean.groupby("year_month")["PM2.5"]
    .mean()
    .reset_index()
)

plt.figure(figsize=(16, 6))
sns.lineplot(data=monthly_all, x="year_month", y="PM2.5", linewidth=2)
plt.title("Tren Rata-rata PM2.5 Bulanan Seluruh Stasiun")
plt.xlabel("Bulan")
plt.ylabel("Rata-rata PM2.5")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


**Interpretasi:** gunakan ranking stasiun untuk mengidentifikasi lokasi dengan konsentrasi PM2.5 rata-rata tertinggi. Grafik tren bulanan menunjukkan periode ketika konsentrasi PM2.5 meningkat atau menurun secara agregat. Nilai final pada kesimpulan harus mengikuti output notebook setelah 12 CSV dijalankan.


### Pertanyaan 2
**Bagaimana hubungan kondisi meteorologi (TEMP, PRES, DEWP, RAIN, WSPM) dengan rata-rata PM2.5 selama periode pengamatan?**


In [ ]:
# Visualisasi 3: heatmap korelasi
plt.figure(figsize=(8, 6))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="vlag",
    center=0,
    square=True
)
plt.title("Korelasi PM2.5 dengan Variabel Meteorologi")
plt.tight_layout()
plt.show()


In [ ]:
# Visualisasi 4: rata-rata PM2.5 menurut musim
season_order = ["Spring", "Summer", "Autumn", "Winter"]
plt.figure(figsize=(9, 5))
sns.barplot(
    data=season_pm25.reset_index(),
    x="season",
    y="mean",
    order=season_order,
    errorbar=None
)
plt.title("Rata-rata PM2.5 Menurut Musim")
plt.xlabel("Musim")
plt.ylabel("Rata-rata PM2.5")
plt.tight_layout()
plt.show()


**Interpretasi:** korelasi digunakan untuk melihat arah dan kekuatan hubungan linear, bukan untuk menyatakan sebab-akibat. Perbandingan musiman melengkapi korelasi dengan menunjukkan periode musim yang secara agregat memiliki PM2.5 lebih tinggi.


## Analisis Lanjutan

### Manual Grouping / Binning Tingkat Polusi

Sebagai analisis lanjutan yang relevan dengan dataset, konsentrasi PM2.5 dikelompokkan ke dalam tiga kategori eksploratif berdasarkan ambang yang transparan. Kategori ini digunakan untuk mempermudah komunikasi risiko relatif di dashboard, bukan sebagai standar regulasi resmi.


In [ ]:
# Kategori eksploratif PM2.5
bins = [-np.inf, 35, 75, np.inf]
labels = ["Rendah (≤35)", "Sedang (36–75)", "Tinggi (>75)"]

df_clean["PM25_category"] = pd.cut(
    df_clean["PM2.5"],
    bins=bins,
    labels=labels
)

category_summary = (
    df_clean["PM25_category"]
    .value_counts(normalize=True)
    .reindex(labels)
    .mul(100)
    .round(2)
    .rename("percentage")
    .reset_index()
    .rename(columns={"index": "PM25_category"})
)

display(category_summary)

station_category = pd.crosstab(
    df_clean["station"],
    df_clean["PM25_category"],
    normalize="index"
).mul(100).round(2)

display(station_category)


In [ ]:
# Hari/jam dengan konsentrasi PM2.5 tinggi sebagai insight operasional
high_pm25 = (
    df_clean.groupby(["year_month", "station"])["PM2.5"]
    .mean()
    .reset_index()
    .sort_values("PM2.5", ascending=False)
    .head(10)
)

display(high_pm25)


## Conclusion & Recommendation

### Kesimpulan
1. **Pertanyaan 1:** ranking stasiun dan tren bulanan digunakan untuk menentukan stasiun dengan rata-rata PM2.5 tertinggi serta periode waktu dengan konsentrasi paling tinggi. Nilai dan nama stasiun final diambil langsung dari output `station_ranking` setelah notebook dijalankan.
2. **Pertanyaan 2:** matriks korelasi menunjukkan arah hubungan PM2.5 dengan TEMP, PRES, DEWP, RAIN, dan WSPM; analisis musiman membantu mengidentifikasi musim dengan rata-rata PM2.5 paling tinggi. Korelasi tidak diartikan sebagai hubungan sebab-akibat.

### Rekomendasi Action Item
- Prioritaskan pemantauan dan evaluasi kualitas udara pada stasiun yang berada di peringkat teratas rata-rata PM2.5.
- Gunakan tren bulanan untuk meningkatkan kesiapsiagaan pada periode yang secara historis memiliki konsentrasi PM2.5 tinggi.
- Integrasikan informasi meteorologi, terutama variabel yang memiliki hubungan paling kuat dengan PM2.5, ke dalam pemantauan dan komunikasi risiko.
- Gunakan dashboard untuk memeriksa perubahan berdasarkan stasiun, periode, dan musim sebelum mengambil keputusan operasional.


## Catatan Reproducibility

Notebook ini sengaja tidak mengarang angka hasil karena 12 CSV dataset belum disertakan dalam upload percakapan. Setelah 12 CSV ditempatkan di `data/`, **Run All** akan menghasilkan angka, tabel, grafik, dan insight berbasis data aktual. Struktur notebook mengikuti kriteria submission: pertanyaan bisnis, gathering, assessing, cleaning, EDA, visualization/explanatory analysis, analisis lanjutan, serta conclusion & recommendation.
